In [14]:
import numpy as np
import pandas as pd
import os

In [15]:
df = pd.read_parquet(r"d:\data_proj\data\processed\clean_retail.parquet")
print(f"Loaded {len(df):,} clean rows!")
df.head()

Loaded 805,549 clean rows!


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [16]:
# 1. Set snapshot date (Dec 10, 2011)
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Snapshot Date for analysis: {snapshot_date}")

# 2. Group by Customer ID and calculate RFM
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency in days
    'Invoice': 'nunique',                                     # Frequency (unique orders)
    'TotalPrice': 'sum'                                       # Monetary (total spent)
}).reset_index()

# 3. Rename columns
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print(f"Total customers profiled: {len(rfm):,}")
rfm.head()

Snapshot Date for analysis: 2011-12-10 12:50:00
Total customers profiled: 5,878


,CustomerID,Recency,Frequency,Monetary
0,12346,326,12,77556.46
1,12347,2,8,5633.32
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40


In [ ]:
# 1. Score 1 to 5 using quintiles
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RF_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str)

# 2. Map to Human Business Segments
def assign_segment(row):
    r, f = row['R_Score'], row['F_Score']
    if r in [4, 5] and f in [4, 5]: return 'Champions'
    elif r in [2, 3, 4, 5] and f in [3, 4, 5]: return 'Loyal Customers'
    elif r in [3, 4, 5] and f in [2, 3]: return 'Potential Loyalists'
    elif r in [4, 5] and f == 1: return 'New Customers'
    elif r == 3 and f == 1: return 'Promising'
    elif r in [2, 3] and f == 2: return 'Need Attention'
    elif r in [1, 2] and f in [3, 4]: return 'At Risk (High Value)'
    elif r == 1 and f == 5: return 'Cant Lose Them'
    elif r == 2 and f == 1: return 'About to Sleep'
    else: return 'Lost / Hibernating'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

# 3. Save to Parquet
rfm.to_parquet(r'd:\data_proj\data\processed\rfm_customers.parquet', index=False)
print('Saved rfm_customers.parquet successfully!')
rfm['Segment'].value_counts()

In [ ]:
# Executive Business Summary by Segment
summary = rfm.groupby('Segment').agg({
    'CustomerID': 'count',
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'sum']
}).round(1)

summary.columns = ['Customer_Count', 'Avg_Recency_Days', 'Avg_Orders', 'Avg_Spend_£', 'Total_Revenue_£']
summary['Revenue_Share_%'] = ((summary['Total_Revenue_£'] / summary['Total_Revenue_£'].sum()) * 100).round(1)
summary = summary.sort_values(by='Total_Revenue_£', ascending=False)
display(summary)
